# 04 — Weekly Team Strength

This notebook updates team strength sequentially using the new in season evidence created in `03_Inseason_Feature_Engineering.ipynb`.

The original preseason model remains unchanged.

The weekly model follows a sequential principle:

> The previous week's team strength is the prior. The most recently completed week provides new evidence.

For each target week, this notebook:

- Loads the frozen preseason ratings for reference
- Loads the previous week's team strength state
- Loads the target week's new in season feature table
- Applies the opponent adjusted weekly strength update
- Preserves separate offensive and defensive weekly signals
- Calculates current weekly rankings
- Compares rank and strength movement to the previous week
- Saves a separate weekly team strength file

No original preseason file or prior weekly file is overwritten.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


## Paths and Weekly Settings

This notebook expects the output from:

`03_Inseason_Feature_Engineering.ipynb`

For Week 3:

- `TARGET_WEEK = 3`
- `PRIOR_WEEK = 2`

The Week 2 team strength file becomes the entering prior for Week 3.

In [2]:
PROJECT_ROOT = Path("../..")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
WEEKLY_DATA_DIR = PROCESSED_DIR / "weekly"

SEASON = 2026
TARGET_WEEK = 3
PRIOR_WEEK = TARGET_WEEK - 1

PRESEASON_STRENGTH_PATH = (
    PROCESSED_DIR
    / "2026_team_strength.parquet"
)

PRIOR_WEEK_STRENGTH_PATH = (
    WEEKLY_DATA_DIR
    / f"week_{PRIOR_WEEK:02d}_team_strength.parquet"
)

INSEASON_FEATURES_PATH = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_inseason_features.parquet"
)

OUTPUT_PATH = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_team_strength.parquet"
)

print("Preseason ratings:", PRESEASON_STRENGTH_PATH)
print("Prior-week ratings:", PRIOR_WEEK_STRENGTH_PATH)
print("New in-season features:", INSEASON_FEATURES_PATH)
print("Weekly output:", OUTPUT_PATH)

Preseason ratings: ..\..\data\processed\2026_team_strength.parquet
Prior-week ratings: ..\..\data\processed\weekly\week_02_team_strength.parquet
New in-season features: ..\..\data\processed\weekly\week_03_inseason_features.parquet
Weekly output: ..\..\data\processed\weekly\week_03_team_strength.parquet


# Load Frozen Preseason Team Strength

The preseason team strength file is the final output of the original `05_Team_Strength_Model.ipynb`.

Its `team_strength` value is already expressed in point differential terms, so it can serve directly as the prior for weekly updates.

We immediately rename the important preseason columns so it is impossible to confuse the original ratings with the new weekly ratings.


In [3]:
preseason = pd.read_parquet(
    PRESEASON_STRENGTH_PATH
).copy()

required_preseason_columns = [
    "team",
    "team_strength",
    "baseline_team_strength",
    "personnel_adjustment",
    "personnel_strength",
    "roster_continuity",
    "roster_continuity_adjustment"
]

missing_preseason = [
    column
    for column in required_preseason_columns
    if column not in preseason.columns
]

if missing_preseason:
    raise KeyError(
        "Missing required preseason columns: "
        + ", ".join(missing_preseason)
    )

preseason = preseason.rename(
    columns={
        "strength_rank": "preseason_strength_rank",
        "team_strength": "preseason_team_strength",
        "baseline_team_strength": "preseason_baseline_team_strength",
        "personnel_adjustment": "preseason_personnel_adjustment",
        "personnel_strength": "preseason_personnel_strength",
        "roster_continuity": "preseason_roster_continuity",
        "roster_continuity_adjustment": "preseason_roster_continuity_adjustment"
    }
)

print("Preseason teams:", len(preseason))
print("Unique teams:", preseason["team"].nunique())

display(
    preseason[
        [
            "preseason_strength_rank",
            "team",
            "preseason_team_strength",
            "preseason_baseline_team_strength",
            "preseason_personnel_adjustment",
            "preseason_roster_continuity_adjustment"
        ]
    ].sort_values(
        "preseason_strength_rank"
    )
)


Preseason teams: 32
Unique teams: 32


,preseason_strength_rank,team,preseason_team_strength,preseason_baseline_team_strength,preseason_personnel_adjustment,preseason_roster_continuity_adjustment
0,1,BUF,4.816514,4.125764,0.934506,0.260091
1,2,LA,4.762452,3.257133,2.100811,0.489665
2,3,SEA,4.363421,3.515976,0.635683,0.932073
3,4,DET,4.018565,3.374644,1.041923,0.037533
4,5,DEN,3.569507,2.355154,0.979753,1.253003
5,6,BAL,3.074111,2.710864,0.636254,-0.037012
6,7,HOU,2.810133,2.193179,1.173833,-0.174691
7,8,GB,2.536317,1.718369,0.882628,0.576742
8,9,PHI,2.451919,2.324982,0.264690,-0.063756
9,10,SF,2.366788,1.805709,0.692939,0.290631


# Load Prior-Week Team Strength

The previous week's team strength file is the starting point for the new weekly update.

For Week 3, the model begins with the Week 2 ratings.

This prevents older games from being counted repeatedly and creates a true sequential rating system.

In [4]:
prior_week = pd.read_parquet(
    PRIOR_WEEK_STRENGTH_PATH
).copy()

required_prior_columns = [
    "team",
    "weekly_strength_rank",
    "weekly_team_strength"
]

missing_prior = [
    column
    for column in required_prior_columns
    if column not in prior_week.columns
]

if missing_prior:
    raise KeyError(
        "Missing required prior-week columns: "
        + ", ".join(missing_prior)
    )

prior_week = prior_week[
    required_prior_columns
].rename(
    columns={
        "weekly_strength_rank": "prior_week_strength_rank",
        "weekly_team_strength": "prior_week_team_strength"
    }
)

print(
    f"Week {PRIOR_WEEK} teams:",
    len(prior_week)
)

print(
    "Unique teams:",
    prior_week["team"].nunique()
)

display(
    prior_week.sort_values(
        "prior_week_strength_rank"
    )
)

Week 2 teams: 32
Unique teams: 32


,team,prior_week_strength_rank,prior_week_team_strength
0,BUF,1,5.233180
1,SEA,2,4.613421
2,BAL,3,4.574111
3,DET,4,4.101898
4,SF,5,4.033455
5,JAX,6,3.713136
6,LA,7,3.095785
7,KC,8,3.052244
8,PHI,9,2.618585
9,HOU,10,2.393466


# Load New Weekly In-Season Features

These features contain the new evidence generated in notebook `03`.

For Week 3, they describe each team's Week 2 performance relative to its entering Week 2 expectation.

The primary values used here are:

- `weekly_strength_update`
- `weekly_offense_adjustment`
- `weekly_defense_adjustment`

The overall strength update is opponent adjusted and uses diminishing returns for extreme single game results.

In [5]:
inseason = pd.read_parquet(
    INSEASON_FEATURES_PATH
).copy()

required_inseason_columns = [
    "team",
    "games_in_update",
    "entering_team_strength",
    "entering_strength_rank",
    "average_opponent_strength",
    "average_opponent_rank",
    "actual_margin",
    "expected_margin",
    "performance_residual",
    "weekly_strength_update",
    "weekly_offense_adjustment",
    "weekly_defense_adjustment"
]

missing_inseason = [
    column
    for column in required_inseason_columns
    if column not in inseason.columns
]

if missing_inseason:
    raise KeyError(
        "Missing required in-season columns: "
        + ", ".join(missing_inseason)
    )

print("In-season teams:", len(inseason))
print(
    "Unique teams:",
    inseason["team"].nunique()
)

display(
    inseason[
        required_inseason_columns
    ].sort_values(
        "weekly_strength_update",
        ascending=False
    )
)

In-season teams: 32
Unique teams: 32


,team,games_in_update,entering_team_strength,entering_strength_rank,average_opponent_strength,average_opponent_rank,actual_margin,expected_margin,performance_residual,weekly_strength_update,weekly_offense_adjustment,weekly_defense_adjustment
4,CAR,1,-6.366575,31,-1.914606,23.0,31.0,-5.951970,36.951970,1.484783,1.995390,2.511015
27,SEA,1,4.613421,2,-2.678016,25.0,24.0,5.791437,18.208563,1.292821,1.516762,1.882387
6,CIN,1,-0.239434,19,2.393466,10.0,14.0,-4.132900,18.132900,1.290726,0.056942,2.222567
18,LV,1,-4.526676,28,-0.089319,18.0,12.0,-5.937356,17.937356,1.285222,0.863838,0.929463
21,NE,1,1.660188,14,1.127236,15.0,17.0,2.032952,14.967048,1.183669,0.009459,2.625084
22,NO,1,-1.639387,22,4.574111,3.0,7.0,-7.713498,14.713498,1.173271,0.738717,0.654342
16,LA,1,3.095785,7,-2.789195,26.0,22.0,7.384980,14.615020,1.169153,1.062593,2.028218
8,DAL,1,-1.053063,21,-2.168535,24.0,17.0,2.615472,14.384528,1.159335,2.435867,-0.048508
28,SF,1,4.033455,5,-3.609921,27.0,22.0,9.143376,12.856624,1.087660,2.081815,0.947440
7,CLE,1,-6.150395,30,0.101026,17.0,4.0,-7.751421,11.751421,1.028227,0.420976,0.186601


# Merge Prior Week State with New Evidence

The weekly model now combines three pieces of information:

1. Frozen preseason information for long term reference
2. Previous week's team strength as the current prior
3. Most recent week's opponent adjusted performance as new evidence

All 32 teams must remain in the table.

In [6]:
weekly_strength = (
    preseason
    .merge(
        prior_week,
        on="team",
        how="left",
        validate="one_to_one"
    )
    .merge(
        inseason,
        on="team",
        how="left",
        validate="one_to_one"
    )
)

weekly_fill_columns = [
    "games_in_update",
    "weekly_strength_update",
    "weekly_offense_adjustment",
    "weekly_defense_adjustment"
]

weekly_strength[
    weekly_fill_columns
] = (
    weekly_strength[
        weekly_fill_columns
    ]
    .fillna(0.0)
)

print("Merged teams:", len(weekly_strength))
print(
    "Unique teams:",
    weekly_strength["team"].nunique()
)

print(
    "Missing prior-week strengths:",
    weekly_strength[
        "prior_week_team_strength"
    ].isna().sum()
)

Merged teams: 32
Unique teams: 32
Missing prior-week strengths: 0


# Sequential Weekly Rating Update

The previous week's team strength is the starting point.

The current week's opponent adjusted performance update is then applied:

`weekly_team_strength = prior_week_team_strength + weekly_strength_update`

Therefore, for Week 3:

`Week 3 Strength = Week 2 Strength + Week 2 Performance Update`

Older results remain embedded in the prior week rating and are not counted again.

The offensive and defensive weekly adjustments remain available separately for the weekly scoring model.

In [7]:
weekly_strength["team_strength_change"] = (
    weekly_strength["weekly_strength_update"]
)

weekly_strength["weekly_team_strength"] = (
    weekly_strength["prior_week_team_strength"]
    + weekly_strength["team_strength_change"]
)

display(
    weekly_strength[
        [
            "team",
            "prior_week_team_strength",
            "weekly_offense_adjustment",
            "weekly_defense_adjustment",
            "team_strength_change",
            "weekly_team_strength"
        ]
    ].sort_values(
        "weekly_team_strength",
        ascending=False
    )
)

,team,prior_week_team_strength,weekly_offense_adjustment,weekly_defense_adjustment,team_strength_change,weekly_team_strength
0,BUF,5.233180,3.271009,-1.463366,0.723864,5.957044
2,SEA,4.613421,1.516762,1.882387,1.292821,5.906242
9,SF,4.033455,2.081815,0.947440,1.087660,5.121115
1,LA,3.095785,1.062593,2.028218,1.169153,4.264938
5,BAL,4.574111,-0.544290,-0.628665,-1.173271,3.400840
3,DET,4.101898,1.813432,-2.920943,-0.723864,3.378034
11,JAX,3.713136,-1.014581,0.101044,-0.725910,2.987226
10,NE,1.660188,0.009459,2.625084,1.183669,2.843857
12,KC,3.052244,1.880896,-1.503479,-0.267123,2.785121
17,MIN,1.891707,-1.616010,2.649615,0.725587,2.617294


# Updated Weekly Rankings

The current weekly rank is based on `weekly_team_strength`.

Ranking movement is now measured relative to the immediately previous week rather than the preseason ranking.

A positive `rank_change` means the team moved up.

For example:

- Week 2 rank = 8
- Week 3 rank = 5
- Rank change = +3

In [8]:
weekly_strength = weekly_strength.sort_values(
    [
        "weekly_team_strength",
        "prior_week_team_strength"
    ],
    ascending=[False, False]
).reset_index(drop=True)

weekly_strength["weekly_strength_rank"] = (
    np.arange(
        1,
        len(weekly_strength) + 1
    )
)

weekly_strength["rank_change"] = (
    weekly_strength["prior_week_strength_rank"]
    - weekly_strength["weekly_strength_rank"]
)

weekly_comparison_table = (
    weekly_strength[
        [
            "weekly_strength_rank",
            "team",
            "weekly_team_strength",
            "prior_week_strength_rank",
            "prior_week_team_strength",
            "team_strength_change",
            "rank_change"
        ]
    ]
    .copy()
)

display(weekly_comparison_table)

,weekly_strength_rank,team,weekly_team_strength,prior_week_strength_rank,prior_week_team_strength,team_strength_change,rank_change
0,1,BUF,5.957044,1,5.233180,0.723864,0
1,2,SEA,5.906242,2,4.613421,1.292821,0
2,3,SF,5.121115,5,4.033455,1.087660,2
3,4,LA,4.264938,7,3.095785,1.169153,3
4,5,BAL,3.400840,3,4.574111,-1.173271,-2
5,6,DET,3.378034,4,4.101898,-0.723864,-2
6,7,JAX,2.987226,6,3.713136,-0.725910,-1
7,8,NE,2.843857,14,1.660188,1.183669,6
8,9,KC,2.785121,8,3.052244,-0.267123,-1
9,10,MIN,2.617294,11,1.891707,0.725587,1


# Biggest Weekly Movers

This table is diagnostic only.

It shows which teams experienced the largest overall rating changes from the previous week to the current week.

The underlying projection model still retains the full preseason, offense, defense, opponent and weekly information.

In [9]:
mover_columns = [
    "team",
    "prior_week_team_strength",
    "team_strength_change",
    "weekly_team_strength",
    "prior_week_strength_rank",
    "weekly_strength_rank",
    "rank_change"
]

biggest_risers = (
    weekly_strength[
        mover_columns
    ]
    .sort_values(
        "team_strength_change",
        ascending=False
    )
    .head(10)
)

biggest_fallers = (
    weekly_strength[
        mover_columns
    ]
    .sort_values(
        "team_strength_change",
        ascending=True
    )
    .head(10)
)

print("BIGGEST RISERS")
display(biggest_risers)

print()
print("BIGGEST FALLERS")
display(biggest_fallers)

BIGGEST RISERS


,team,prior_week_team_strength,team_strength_change,weekly_team_strength,prior_week_strength_rank,weekly_strength_rank,rank_change
29,CAR,-6.366575,1.484783,-4.881793,31,30,1
1,SEA,4.613421,1.292821,5.906242,2,2,0
14,CIN,-0.239434,1.290726,1.051292,19,15,4
22,LV,-4.526676,1.285222,-3.241453,28,23,5
7,NE,1.660188,1.183669,2.843857,14,8,6
18,NO,-1.639387,1.173271,-0.466115,22,19,3
3,LA,3.095785,1.169153,4.264938,7,4,3
16,DAL,-1.053063,1.159335,0.106272,21,17,4
2,SF,4.033455,1.087660,5.121115,5,3,2
30,CLE,-6.150395,1.028227,-5.122168,30,31,-1



BIGGEST FALLERS


,team,prior_week_team_strength,team_strength_change,weekly_team_strength,prior_week_strength_rank,weekly_strength_rank,rank_change
24,ATL,-1.914606,-1.484783,-3.399388,23,25,-2
26,ARI,-2.678016,-1.292821,-3.970836,25,27,-2
12,HOU,2.393466,-1.290726,1.102740,10,13,-3
21,LAC,-0.089319,-1.285222,-1.374541,18,22,-4
17,PIT,1.127236,-1.183669,-0.056433,15,18,-3
4,BAL,4.574111,-1.173271,3.400840,3,5,-2
25,NYG,-2.789195,-1.169153,-3.958348,26,26,0
23,WAS,-2.168535,-1.159335,-3.327871,24,24,0
28,MIA,-3.609921,-1.087660,-4.697581,27,29,-2
20,TB,0.101026,-1.028227,-0.927200,17,21,-4


# Update Diagnostics

These diagnostics check the size and distribution of the sequential weekly update.

The comparison is now:

`Prior Week Strength → Current Week Strength`

rather than:

`Preseason Strength → Current Week Strength`

In [10]:
movement = (
    weekly_strength[
        "team_strength_change"
    ].abs()
)

print(
    f"WEEK {PRIOR_WEEK} → "
    f"WEEK {TARGET_WEEK} TEAM-STRENGTH MOVEMENT"
)

print(
    weekly_strength[
        "team_strength_change"
    ]
    .describe()
    .round(3)
)

print()

print(
    "Mean absolute movement:",
    round(
        movement.mean(),
        3
    )
)

print(
    "Maximum absolute movement:",
    round(
        movement.max(),
        3
    )
)

print(
    "Team with largest movement:",
    weekly_strength.loc[
        movement.idxmax(),
        "team"
    ]
)

print()
print("RATING DISTRIBUTIONS")

distribution_check = pd.DataFrame(
    {
        "metric": [
            f"Week {PRIOR_WEEK} Team Strength",
            f"Week {TARGET_WEEK} Team Strength"
        ],
        "mean": [
            weekly_strength[
                "prior_week_team_strength"
            ].mean(),
            weekly_strength[
                "weekly_team_strength"
            ].mean()
        ],
        "std": [
            weekly_strength[
                "prior_week_team_strength"
            ].std(),
            weekly_strength[
                "weekly_team_strength"
            ].std()
        ],
        "min": [
            weekly_strength[
                "prior_week_team_strength"
            ].min(),
            weekly_strength[
                "weekly_team_strength"
            ].min()
        ],
        "max": [
            weekly_strength[
                "prior_week_team_strength"
            ].max(),
            weekly_strength[
                "weekly_team_strength"
            ].max()
        ]
    }
)

display(
    distribution_check.round(3)
)

WEEK 2 → WEEK 3 TEAM-STRENGTH MOVEMENT
count    32.000
mean      0.000
std       1.043
min      -1.485
25%      -1.106
50%       0.000
75%       1.106
max       1.485
Name: team_strength_change, dtype: float64

Mean absolute movement: 0.954
Maximum absolute movement: 1.485
Team with largest movement: ATL

RATING DISTRIBUTIONS


,metric,mean,std,min,max
0,Week 2 Team Strength,0.002,3.539,-7.831,5.233
1,Week 3 Team Strength,0.002,3.594,-7.321,5.957


# Sanity Checks

These checks verify the sequential weekly update.

The weekly file should:

- contain exactly 32 unique teams
- preserve the frozen preseason information
- contain a valid prior week rating for every team
- contain no missing current weekly strength values
- have `team_strength_change` equal to the new opponent adjusted weekly update
- calculate current strength as prior week strength plus the weekly update
- never overwrite the preseason or prior week files

In [11]:
assert len(weekly_strength) == 32, (
    "Expected 32 teams in weekly strength table."
)

assert (
    weekly_strength["team"].nunique()
    == 32
), (
    "Expected 32 unique teams."
)

assert weekly_strength[
    "preseason_team_strength"
].notna().all(), (
    "Missing frozen preseason ratings."
)

assert weekly_strength[
    "prior_week_team_strength"
].notna().all(), (
    "Missing prior-week ratings."
)

assert weekly_strength[
    "weekly_team_strength"
].notna().all(), (
    "Missing current weekly ratings."
)

assert np.allclose(
    weekly_strength[
        "team_strength_change"
    ],
    weekly_strength[
        "weekly_strength_update"
    ]
), (
    "Team-strength change does not match "
    "the opponent-adjusted weekly update."
)

assert np.allclose(
    weekly_strength[
        "weekly_team_strength"
    ],
    (
        weekly_strength[
            "prior_week_team_strength"
        ]
        + weekly_strength[
            "team_strength_change"
        ]
    )
), (
    "Sequential weekly strength formula "
    "is inconsistent."
)

assert np.allclose(
    weekly_strength[
        "entering_team_strength"
    ],
    weekly_strength[
        "prior_week_team_strength"
    ]
), (
    "Notebook 03 entering strength does not "
    "match the prior-week team strength."
)

print(
    "All sequential weekly team-strength "
    "sanity checks passed."
)

All sequential weekly team-strength sanity checks passed.


# Final Weekly Team Strength Table

The saved file contains the full state needed by the weekly projection pipeline.

This includes:

- current weekly strength and rank
- previous week's strength and rank
- week to week movement
- frozen preseason information
- opponent adjusted performance information
- separate offensive and defensive weekly signals

The compact ranking table shown earlier is only for interpretation. The saved dataset retains the additional model information.

In [12]:
final_columns = [
    "weekly_strength_rank",
    "team",
    "weekly_team_strength",

    "prior_week_strength_rank",
    "prior_week_team_strength",
    "team_strength_change",
    "rank_change",

    "games_in_update",
    "average_opponent_rank",
    "average_opponent_strength",
    "actual_margin",
    "expected_margin",
    "performance_residual",
    "weekly_strength_update",

    "weekly_offense_adjustment",
    "weekly_defense_adjustment",

    "preseason_strength_rank",
    "preseason_team_strength",
    "preseason_baseline_team_strength",
    "preseason_personnel_adjustment",
    "preseason_personnel_strength",
    "preseason_roster_continuity",
    "preseason_roster_continuity_adjustment"
]

weekly_team_strength = (
    weekly_strength[
        final_columns
    ]
    .copy()
)

display(
    weekly_team_strength
)

,weekly_strength_rank,team,weekly_team_strength,prior_week_strength_rank,prior_week_team_strength,team_strength_change,rank_change,games_in_update,average_opponent_rank,average_opponent_strength,...,weekly_strength_update,weekly_offense_adjustment,weekly_defense_adjustment,preseason_strength_rank,preseason_team_strength,preseason_baseline_team_strength,preseason_personnel_adjustment,preseason_personnel_strength,preseason_roster_continuity,preseason_roster_continuity_adjustment
0,1,BUF,5.957044,1,5.233180,0.723864,0,1,4.0,4.101898,...,0.723864,3.271009,-1.463366,1,4.816514,4.125764,0.934506,0.623004,0.573034,0.260091
1,2,SEA,5.906242,2,4.613421,1.292821,0,1,25.0,-2.678016,...,1.292821,1.516762,1.882387,3,4.363421,3.515976,0.635683,0.423788,0.677419,0.932073
2,3,SF,5.121115,5,4.033455,1.087660,2,1,27.0,-3.609921,...,1.087660,2.081815,0.947440,10,2.366788,1.805709,0.692939,0.461959,0.577778,0.290631
3,4,LA,4.264938,7,3.095785,1.169153,3,1,26.0,-2.789195,...,1.169153,1.062593,2.028218,2,4.762452,3.257133,2.100811,1.400540,0.608696,0.489665
4,5,BAL,3.400840,3,4.574111,-1.173271,-2,1,22.0,-1.639387,...,-1.173271,-0.544290,-0.628665,6,3.074111,2.710864,0.636254,0.424170,0.526882,-0.037012
5,6,DET,3.378034,4,4.101898,-0.723864,-2,1,1.0,5.233180,...,-0.723864,1.813432,-2.920943,4,4.018565,3.374644,1.041923,0.694616,0.538462,0.037533
6,7,JAX,2.987226,6,3.713136,-0.725910,-1,1,12.0,1.819507,...,-0.725910,-1.014581,0.101044,12,1.713136,1.746953,-0.223729,-0.149153,0.563830,0.200841
7,8,NE,2.843857,14,1.660188,1.183669,6,1,15.0,1.127236,...,1.183669,0.009459,2.625084,11,1.910188,1.605622,0.906186,0.604124,0.458333,-0.478291
8,9,KC,2.785121,8,3.052244,-0.267123,-1,1,20.0,-0.967774,...,-0.267123,1.880896,-1.503479,13,1.302244,1.484935,-0.187743,-0.125162,0.510870,-0.140090
9,10,MIN,2.617294,11,1.891707,0.725587,1,1,13.0,1.781399,...,0.725587,-1.616010,2.649615,18,0.475040,0.919610,-0.620638,-0.413759,0.510204,-0.144374


# Save Weekly Team Strength

This writes only to:

`data/processed/weekly/`

The original:

`data/processed/2026_team_strength.parquet`

remains untouched.


In [13]:
OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

weekly_team_strength.to_parquet(
    OUTPUT_PATH,
    index=False
)

print("Saved:", OUTPUT_PATH)
print()
print(
    "Original preseason file remains:",
    PRESEASON_STRENGTH_PATH
)


Saved: ..\..\data\processed\weekly\week_03_team_strength.parquet

Original preseason file remains: ..\..\data\processed\2026_team_strength.parquet


# What This Means for Week 3

At the end of this notebook, every team has:

- its frozen preseason rating
- its Week 2 entering rating and rank
- its opponent adjusted Week 2 performance
- its Week 2 offensive and defensive signals
- its Week 2 → Week 3 rating movement
- its updated Week 3 team strength rating
- its updated Week 3 rank

The next stage will apply the Week 3 injury update and then use the resulting team state to generate Week 3 game projections.

The weekly ranking table is diagnostic only.

The saved weekly team strength file retains the additional information required by the projection pipeline.